# PRICE RANGE PREDICTION

# ============================================================
# PRICE RANGE PREDICTION
# Goal: Predict restaurant price range (1-4) using MLP/DNN
# Type: Multi-class Classification
# ============================================================

In [ ]:

# ---------- 1. Install & Import Libraries ----------
!pip install pandas numpy matplotlib seaborn scikit-learn torch torchmetrics openpyxl


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchmetrics import Accuracy, F1Score
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# ---------- 2. Load Dataset ----------
from google.colab import files
uploaded = files.upload()

df = pd.read_csv('Restaurant Dataset.csv', encoding='latin1')
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# ---------- 3. Data Preprocessing ----------
print("\n" + "="*60)
print("DATA PREPROCESSING")
print("="*60)

# Select relevant columns for price range prediction
selected_cols = [
    'City', 'Locality', 'Cuisines', 'Aggregate rating',
    'Has Table booking', 'Has Online delivery', 'Price range'
]
df_price = df[selected_cols].copy()

# Drop rows with missing target
df_price = df_price.dropna(subset=['Price range'])

# Handle missing values in features
df_price['Cuisines'] = df_price['Cuisines'].fillna('Unknown')
df_price['Locality'] = df_price['Locality'].fillna('Unknown')

# Convert boolean columns to binary (0/1)
df_price['Has Table booking'] = df_price['Has Table booking'].map({'Yes': 1, 'No': 0})
df_price['Has Online delivery'] = df_price['Has Online delivery'].map({'Yes': 1, 'No': 0})

# Fill missing rating values with median
df_price['Aggregate rating'] = df_price['Aggregate rating'].fillna(df_price['Aggregate rating'].median())

In [ ]:
# ---------- 4. Feature Engineering ----------
print("\n--- Feature Engineering ---")

# Extract primary cuisine (first cuisine from comma-separated list)
df_price['Primary Cuisine'] = df_price['Cuisines'].apply(
    lambda x: str(x).split(',')[0].strip() if pd.notna(x) else 'Unknown'
)

# ---------- 5. Encode Categorical Variables ----------
print("\n--- Encoding Categorical Variables ---")

# Label encode categorical features
label_encoders = {}
categorical_cols = ['City', 'Locality', 'Primary Cuisine']

for col in categorical_cols:
    le = LabelEncoder()
    df_price[col] = le.fit_transform(df_price[col])
    label_encoders[col] = le
    print(f"  {col}: {len(le.classes_)} unique classes")

# ---------- 6. Feature & Target Split ----------
X = df_price.drop(columns=['Price range', 'Cuisines'])  # Drop original Cuisines
y = df_price['Price range'] - 1  # Convert to 0-3 for classification

print(f"\nFeatures shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Target distribution:\n{y.value_counts().sort_index()}")

In [ ]:
# ---------- 7. Train-Test Split ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# ---------- 8. Feature Scaling ----------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled successfully!")


In [ ]:



# ---------- 9. Convert to PyTorch Tensors ----------
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

# Create DataLoaders
batch_size = 64
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:


# ---------- 10. Define MLP/DNN Model ----------
class PriceRangeClassifier(nn.Module):
    """
    Multi-Layer Perceptron (MLP) / Deep Neural Network (DNN)
    for multi-class classification of restaurant price ranges.
    """
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], num_classes=4, dropout_rate=0.3):
        super(PriceRangeClassifier, self).__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim

        # Output layer
        layers.append(nn.Linear(prev_dim, num_classes))

        self.network = nn.Sequential(*layers)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.network(x)

# Instantiate model
input_dim = X_train_scaled.shape[1]
num_classes = 4
model = PriceRangeClassifier(
    input_dim=input_dim,
    hidden_dims=[128, 64, 32],
    num_classes=num_classes,
    dropout_rate=0.3
).to(device)

print(f"\nModel Architecture:")
print(f"  Input dimension: {input_dim}")
print(f"  Number of classes: {num_classes}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:


# ---------- 11. Define Loss, Optimizer & Metrics ----------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=10, factor=0.5)

# Metrics
train_acc_metric = Accuracy(task="multiclass", num_classes=num_classes).to(device)
test_acc_metric = Accuracy(task="multiclass", num_classes=num_classes).to(device)

# ---------- 12. Training Loop ----------
epochs = 100
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

print("\n" + "="*60)
print("TRAINING MODEL")
print("="*60)

for epoch in range(epochs):
    # Training phase
    model.train()
    epoch_loss = 0
    epoch_acc = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += train_acc_metric(outputs, batch_y)

    avg_train_loss = epoch_loss / len(train_loader)
    avg_train_acc = epoch_acc / len(train_loader)

    # Validation phase
    model.eval()
    val_loss = 0
    val_acc = 0

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)

            val_loss += loss.item()
            val_acc += test_acc_metric(outputs, batch_y)

    avg_val_loss = val_loss / len(test_loader)
    avg_val_acc = val_acc / len(test_loader)

    # Store metrics
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    train_accuracies.append(avg_train_acc.cpu().numpy())
    val_accuracies.append(avg_val_acc.cpu().numpy())

    # Update learning rate
    scheduler.step(avg_val_loss)

    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Train Acc: {avg_train_acc:.4f} | "
              f"Val Acc: {avg_val_acc:.4f}")

print("\nTraining completed!")

In [ ]:

# ---------- 13. Model Evaluation ----------
print("\n" + "="*60)
print("MODEL EVALUATION")
print("="*60)

# Get predictions
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        outputs = model(batch_X)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.numpy())

y_pred = np.array(all_preds)
y_true = np.array(all_labels)

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# Classification Report
print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred,
                            target_names=['Price Range 1', 'Price Range 2', 'Price Range 3', 'Price Range 4'],
                            zero_division=0))

# ---------- 14. Confusion Matrix ----------
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Range 1', 'Range 2', 'Range 3', 'Range 4'],
            yticklabels=['Range 1', 'Range 2', 'Range 3', 'Range 4'])
plt.title('Confusion Matrix - Price Range Prediction')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:



# ---------- 15. Visualizations ----------
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Training & Validation Loss
axes[0, 0].plot(train_losses, label='Train Loss', color='blue', alpha=0.7)
axes[0, 0].plot(val_losses, label='Validation Loss', color='orange', alpha=0.7)
axes[0, 0].set_xlabel('Epochs')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training & Validation Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Training & Validation Accuracy
axes[0, 1].plot(train_accuracies, label='Train Accuracy', color='green', alpha=0.7)
axes[0, 1].plot(val_accuracies, label='Validation Accuracy', color='red', alpha=0.7)
axes[0, 1].set_xlabel('Epochs')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_title('Training & Validation Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Target Distribution (Actual vs Predicted)
actual_counts = pd.Series(y_true).value_counts().sort_index()
# Ensure pred_counts covers all classes, filling missing with 0
pred_counts = pd.Series(y_pred).value_counts().reindex(actual_counts.index, fill_value=0).sort_index()
x_labels = ['Range 1', 'Range 2', 'Range 3', 'Range 4']

ax = axes[1, 0]
x = np.arange(len(x_labels))
width = 0.35
ax.bar(x - width/2, actual_counts.values, width, label='Actual', color='blue', alpha=0.6)
ax.bar(x + width/2, pred_counts.values, width, label='Predicted', color='orange', alpha=0.6)
ax.set_xlabel('Price Range')
ax.set_ylabel('Count')
ax.set_title('Actual vs Predicted Distribution')
ax.set_xticks(x)
ax.set_xticklabels(x_labels)
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Feature Importance (using model weights as proxy)
if hasattr(model.network[0], 'weight'):
    # Get weights from first layer
    first_layer_weights = model.network[0].weight.data.cpu().numpy()
    # Average absolute weights per feature
    feature_importance = np.mean(np.abs(first_layer_weights), axis=0)
    feature_names = X.columns.tolist()

    # Sort features by importance
    sorted_idx = np.argsort(feature_importance)[::-1]
    sorted_features = [feature_names[i] for i in sorted_idx]
    sorted_importance = [feature_importance[i] for i in sorted_idx]

    # Take top features if many
    top_n = min(10, len(sorted_features))
    axes[1, 1].barh(sorted_features[:top_n], sorted_importance[:top_n], color='teal')
    axes[1, 1].set_xlabel('Feature Importance (Avg |Weight|)')
    axes[1, 1].set_title('Top Feature Importance')
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

# ---------- 16. Class-wise Performance ----------
print("\n" + "="*60)
print("CLASS-WISE PERFORMANCE")
print("="*60)

class_names = ['Price Range 1 (Budget)', 'Price Range 2 (Mid)',
               'Price Range 3 (Premium)', 'Price Range 4 (Luxury)']

class_precision = precision_score(y_true, y_pred, average=None, zero_division=0)
class_recall = recall_score(y_true, y_pred, average=None, zero_division=0)
class_f1 = f1_score(y_true, y_pred, average=None, zero_division=0)

class_df = pd.DataFrame({
    'Class': class_names,
    'Precision': class_precision,
    'Recall': class_recall,
    'F1-Score': class_f1,
    'Support': [np.sum(y_true == i) for i in range(num_classes)]
})
print(class_df.to_string(index=False))

# ---------- 17. Error Analysis ----------
print("\n" + "="*60)
print("ERROR ANALYSIS")
print("="*60)

# Find misclassified samples
misclassified = np.where(y_pred != y_true)[0]
print(f"Total misclassified samples: {len(misclassified)}")
print(f"Misclassification rate: {len(misclassified)/len(y_true)*100:.2f}%")

# Analyze which classes are most confused
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
print("\nNormalized Confusion Matrix (Row-wise):")
print(pd.DataFrame(cm_normalized,
                   index=['Range 1', 'Range 2', 'Range 3', 'Range 4'],
                   columns=['Range 1', 'Range 2', 'Range 3', 'Range 4']))

# ---------- 18. Save Model (Optional) ----------
# torch.save(model.state_dict(), 'price_range_model.pth')
# print("\nModel saved as 'price_range_model.pth'")

In [ ]:


# ---------- 19. Prediction Function ----------
def predict_price_range(features):
    """
    Predict price range for new restaurant features.
    features: dict with keys matching feature columns
    """
    model.eval()
    # Convert features to DataFrame and scale
    feat_df = pd.DataFrame([features])

    # Ensure all features are present
    for col in X.columns:
        if col not in feat_df.columns:
            feat_df[col] = 0  # default value

    feat_df = feat_df[X.columns]  # Reorder columns
    feat_scaled = scaler.transform(feat_df)
    feat_tensor = torch.tensor(feat_scaled, dtype=torch.float32).to(device)

    with torch.no_grad():
        outputs = model(feat_tensor)
        _, pred = torch.max(outputs, 1)

    price_ranges = ['Budget (1)', 'Mid (2)', 'Premium (3)', 'Luxury (4)']
    return price_ranges[pred.item()], pred.item() + 1

# Test prediction function
print("\n--- Test Prediction Function ---")
sample_features = {
    'City': 1,  # Encoded city
    'Locality': 2,  # Encoded locality
    'Primary Cuisine': 3,  # Encoded cuisine
    'Aggregate rating': 4.2,
    'Has Table booking': 1,
    'Has Online delivery': 0
}
pred_label, pred_price = predict_price_range(sample_features)
print(f"Sample features: {sample_features}")
print(f"Predicted: {pred_label} (Range {pred_price})")

In [ ]:


# ============================================================
# PART 20: INTERPRETATIONS & ACTIONABLE RECOMMENDATIONS
# ============================================================

print("\n" + "="*60)
print("INTERPRETATIONS & ACTIONABLE RECOMMENDATIONS")
print("="*60)

# ---------- 20.1 Interpretations ----------
print("\n--- INTERPRETATIONS ---")

print(f"\n1. Model Performance:")
print(f"   - Accuracy: {accuracy*100:.2f}%")
print(f"   - Weighted F1-Score: {f1:.4f}")
print(f"   - The model correctly predicts price range for {accuracy*100:.2f}% of restaurants.")

print("\n2. Class Performance Insights:")
for i, name in enumerate(class_names):
    print(f"   - {name}: F1 = {class_f1[i]:.3f} (Support: {np.sum(y_true == i)})")

print("\n3. Most Confused Classes:")
# Find highest off-diagonal confusion
max_off_diag = 0
max_pair = None
for i in range(num_classes):
    for j in range(num_classes):
        if i != j and cm_normalized[i, j] > max_off_diag:
            max_off_diag = cm_normalized[i, j]
            max_pair = (i, j)
if max_pair:
    print(f"   - {class_names[max_pair[0]]} is most often misclassified as {class_names[max_pair[1]]}")
    print(f"   - Confusion rate: {max_off_diag*100:.1f}%")

print("\n4. Key Features Influencing Price Range:")
# Get actual feature importance from weights
first_layer_weights = model.network[0].weight.data.cpu().numpy()
feature_importance = np.mean(np.abs(first_layer_weights), axis=0)
feature_names = X.columns.tolist()
top_3_idx = np.argsort(feature_importance)[::-1][:3]
print(f"   - Top 3 features: {[feature_names[i] for i in top_3_idx]}")

In [ ]:


# ---------- 20.2 Actionable Recommendations ----------
print("\n--- ACTIONABLE RECOMMENDATIONS ---")

print("\n1. Business Applications:")
print("   🏷️ Help new restaurants determine optimal pricing strategy.")
print("   📊 Enable food delivery platforms to categorize restaurants accurately.")
print("   🎯 Assist customers in finding restaurants within their budget.")
print("   📈 Identify market gaps in specific price ranges and locations.")

print("\n2. Model Improvement Strategies:")
print("   ✅ Collect more data with balanced class distribution.")
print("   ✅ Add more features (e.g., location coordinates, nearby attractions).")
print("   ✅ Use embeddings for categorical features (City, Cuisine).")
print("   ✅ Try ensemble methods (Random Forest, XGBoost) for comparison.")
print("   ✅ Implement cross-validation for more robust evaluation.")
print("   ✅ Use SMOTE or class weights to handle class imbalance.")

print("\n3. Deployment Recommendations:")
print("   ✅ Deploy as REST API using FastAPI for real-time predictions.")
print("   ✅ Build interactive dashboard using Streamlit or Gradio.")
print("   ✅ Integrate with restaurant discovery apps and food delivery platforms.")

print("\n4. Feature Engineering Improvements:")
print("   ✅ Add interaction features (e.g., Rating × Table booking).")
print("   ✅ Include more granular location data (neighborhood, area).")
print("   ✅ Extract cuisine categories as multi-label features.")
print("   ✅ Add features like number of votes, reviews sentiment.")

print("\n5. Business Strategy Insights:")
print("   ✅ Identify price range correlations with cuisine types.")
print("   ✅ Analyze how table booking affects perceived value/price.")
print("   ✅ Use insights for targeted marketing and promotions.")
print("   ✅ Help restaurant owners benchmark against similar establishments.")

print("\n" + "="*60)
print("PROJECT COMPLETED SUCCESSFULLY!")
print("="*60)